# A4 open-set verification on Lee 2019 — second-corpus replication

Replicates the PhysioNet AUC 0.925 unseen-subject result on a 54-subject second cohort. Trains a contrastive EEGNet on session_1 of 40 Lee 2019 subjects, then verifies identity on the 14 held-out subjects (whom the network never saw at training time).

Prerequisite: `A3_lee2019_download.ipynb` must have completed (108/108 compact cache on Drive).

**Runtime → L4 GPU.** Expected wall: ~35-45 min.

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
os.environ['BCI_LEE2019_CACHE'] = '/content/drive/MyDrive/bci_cache'
import glob, sys
WIN = '/content/drive/MyDrive/bci_cache/lee2019_mi/windowed'
files = sorted(glob.glob(os.path.join(WIN, '*.npz')))
print(f'Cached: {len(files)}/108')
if len(files) < 108:
    sys.exit('!!! cache incomplete; run A3_lee2019_download.ipynb first.')

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

## Run A4 within-session (40 train / 14 unseen, session_1 → session_1)

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.24_a4_lee2019 --all

## Run A4 cross-session (40 train sess1 / 14 unseen sess2) — bonus harder protocol

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.24_a4_lee2019 --all --cross-session

## Emit run meta + paste-back result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
TAG = 'a4_lee2019'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception:
    git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
outputs = [
    'results/24_a4_lee2019_within_session.json',
    'results/24_a4_lee2019_cross_session.json',
    'figures/24_a4_lee2019_within_session.pdf',
    'figures/24_a4_lee2019_cross_session.pdf',
]
meta = {'run_id': run_id, 'experiment': 'experiments.24_a4_lee2019',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': outputs}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
missing = [p for p in outputs[:2] if not os.path.exists(p)]
if missing:
    sys.exit(f'!!! missing result JSON(s): {missing}. Re-run the experiment cells, then this cell.')
print('--- BEGIN 24_a4_lee2019_within_session.json ---')
print(open('results/24_a4_lee2019_within_session.json').read())
print('--- END 24_a4_lee2019_within_session.json ---')
print()
if os.path.exists('results/24_a4_lee2019_cross_session.json'):
    print('--- BEGIN 24_a4_lee2019_cross_session.json ---')
    print(open('results/24_a4_lee2019_cross_session.json').read())
    print('--- END 24_a4_lee2019_cross_session.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')